# 04 · Create map panels from the saved CML grid and native RADKLIM

**Depends on 03.** No placeholder gridded file anymore. Provide a correct native RADKLIM YW folder through the **same config** (`extras.native_radklim_root`). The original renderer reads daily YW radar files, *not* the path-averaged 5-minute radar product. If the historic event-selection cache exists, it is used; otherwise a reviewer can explicitly supply a demonstration extent, which is never mislabeled as the original event discovery.

The original renderer requires `cartopy`. This step is opt-in, may take a while and never modifies previous files.

In [1]:
from pathlib import Path
import sys, json
# One shared configuration for ALL notebooks (no per-notebook path editing).
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'tools_verify_originals.py').is_file(), f'Wrong notebook working folder: {ROOT}'
sys.path.insert(0,str(ROOT/'src'))
from essd_cz.connected import Project
SETTINGS=json.loads((ROOT/'config/active_profile.json').read_text(encoding='utf-8'))
PROFILE=SETTINGS.get('profile','hpc')
if PROFILE not in ('hpc','user'):raise ValueError("Profile must be 'hpc' or 'user'")
CONFIG=ROOT/('config/hpc_source.example.json' if PROFILE=='hpc' else 'config/user_paths.json')
WORKDIR=ROOT/SETTINGS.get('workspace','results/connected_workflow')
project=Project(CONFIG,WORKDIR)
print('Profile:',PROFILE,'Data config:',project.config,'Output:',project.output)


Profile: hpc Data config: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/config/hpc_source.example.json Output: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow


In [4]:
# Requires cartopy; if missing, install it in this notebook's environment first (pip install cartopy).
from essd_cz.connected import run_native_radar_map
from pathlib import Path
import xarray as xr
GRID=project.get('spatial_grid')
print('Using grid from previous notebook:', GRID)
print('Native RADKLIM root:', project.extras.get('native_radklim_root'))
print('Original event cache:', project.extras.get('historical_event_cache'))
RUN_MAPS=False    # True to execute original map renderer; requires daily native RADKLIM and Cartopy
FIRST_PANEL='2021-07-13 16:05:00'
EVENT_RANK=4
N_COLS=8
PANEL_STEP_MINUTES=60
MANUAL_EXTENT=None  # optional [lonmin, lonmax, latmin, latmax]
ALLOW_MANUAL_DEMONSTRATION=True  # when historical selected-event CSV isn't available
if RUN_MAPS:
    old_cache=project.extras.get('historical_event_cache')
    if (old_cache is None or not old_cache.is_file()) and MANUAL_EXTENT is None and ALLOW_MANUAL_DEMONSTRATION:
        with xr.open_dataset(GRID) as ds:
            MANUAL_EXTENT=[float(ds.x.min()),float(ds.x.max()),float(ds.y.min()),float(ds.y.max())]
        print('Explicit FALLBACK: using grid extent, NOT the paper’s historical event selection.')
    maps=run_native_radar_map(project,first_panel=FIRST_PANEL,ncols=N_COLS,
        step_minutes=PANEL_STEP_MINUTES,event_rank=EVENT_RANK,extent=MANUAL_EXTENT,dpi=200)
    print('Generated map files:',maps)
else:
    print('DRY RUN: prior grid ready. Set RUN_MAPS=True after confirming native RADKLIM YW files.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 46.7 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [cartopy]m1/2 [cartopy]
Using grid from previous notebook: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/03_gridded/reviewer_final_graf_IDW_5min.nc
Native RADKLIM root: /home/jovyan/dwd_radolan/2021/suplement
Original event cache: /home/jovyan/ESSD_publication_dataset/cmlonly_graf_nearby_comparison/apr_oct_2021/article_style/multiple_events_radar/selected_instances_cache.csv
ORIGINAL MAP: /home/jovyan/.venvs/pr13_xarray_2026_04/bin/python /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/original/code/figures/03_blettner_event_maps_timeseries.py --gridded-nc /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/03_gridded/reviewer_final_graf_IDW_5min.nc --cml-var R --cml-row-label GRAF --cml-geom-nc /home/jovyan/ESSD_publication